# Spec 撰寫：把意圖收斂成「規格契約」

## 模組高潮：M2 的典範轉移點——寫規格，不寫長提示詞

本筆記是 **02-意圖收斂** 的高潮。2023–24 的做法是「把 prompt 寫長、堆 few-shot、手刻推理步驟」；但在 gpt-5+ 時代,**強模型過度參數化,冗長指示反而引入雜訊**。2026 的做法是**減法**:

- 把模糊需求收斂成一份短而結構化的 **spec(規格)**——角色、目標、輸入、限制、輸出格式、成功標準。
- spec 是你與模型之間的**契約**:同一份 spec 給不同模型(OpenAI/Claude/Gemini)都應產生結構一致、可驗收的輸出。
- **控制權上移**:隨著模型能力的「泡泡」擴張,人類應從「微調詞彙」退出,改為「定義規格與驗證」——把交接點從**詞彙層**移到**系統層**。

> spec 是後續所有模組的共同起點:輸出格式接 M3(結構)、知識來源接 M4(RAG)、業務規則接 M5(guardrails)、成功標準接 M7(評估)。

## 1. Spec 的七個欄位

一份好的 spec 至少涵蓋:

| 欄位 | 作用 | 對應可控性 |
|------|------|-----------|
| **Role** 角色 | 設定專業視角與語氣 | 收斂風格 |
| **Goal** 目標 | 一句話說清要產出什麼 | 收斂意圖 |
| **Inputs** 輸入 | 明確列出可用素材 | 減少臆測 |
| **Constraints** 限制 | 字數、語言、禁止事項 | 收斂行為邊界 |
| **Output format** 輸出格式 | JSON/Markdown/結構 | 收斂結構(接 M3) |
| **Success criteria** 成功標準 | 可驗收的條件 | 收斂評估(接 M7) |
| **Examples** 範例 | few-shot 示範(格式對齊用) | 收斂風格與格式 |

> 注意:**spec 短而精準**勝過長篇大論。OpenAI 對 gpt-5 的建議框架是 **CTCO(Context→Task→Constraints→Output)**,與這七欄位同源——先給脈絡與任務,再給約束與輸出規格,然後**讓開、讓模型發揮**。

## 2. 環境設定

In [ ]:
from dotenv import load_dotenv
import os, json
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()  # 讀取 OPENAI_API_KEY
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

## 3. 可重用的 Spec 模板

用 Python 函式把七欄位組成一個結構化 system prompt——這就是你的「spec 產生器」。

In [ ]:
def build_spec(role, goal, inputs, constraints, output_format, success_criteria, examples=None):
    """把需求七要素組成一份結構化 spec(system prompt)。"""
    parts = [
        f"# 角色\n{role}",
        f"# 目標\n{goal}",
        f"# 可用輸入\n{inputs}",
        f"# 限制\n{constraints}",
        f"# 輸出格式\n{output_format}",
        f"# 成功標準\n{success_criteria}",
    ]
    if examples:
        parts.append(f"# 範例\n{examples}")
    return "\n\n".join(parts)

spec = build_spec(
    role="你是一位資深 DTC 電商文案,擅長精簡有力的產品標題。",
    goal="為輸入的產品,產生 3 個吸睛的繁體中文標題。",
    inputs="使用者會提供產品名稱與賣點。",
    constraints="每個標題 ≤ 20 字;不得誇大療效;繁體中文。",
    output_format='回傳 JSON:{"titles": ["...", "...", "..."]}',
    success_criteria="3 個標題、皆 ≤20 字、皆與賣點相關。",
    examples='產品:保溫瓶 / 賣點:12 小時保溫 → {"titles": ["12 小時還燙口", ...]}',
)
print(spec)

## 4. 用 spec 驅動模型(OpenAI Responses)

spec 當 system 指令,使用者只需丟最精簡的輸入。輸出受 spec 約束、可直接解析。

In [ ]:
def run_with_spec_openai(spec, user_input, model=OPENAI_MODEL):
    resp = openai_client.responses.create(
        model=model,
        input=[{"role": "system", "content": spec},
               {"role": "user", "content": user_input}],
        temperature=0.7,
        text={"format": {"type": "json_object"}},
    )
    return json.loads(resp.output_text)

out = run_with_spec_openai(spec, "產品:人體工學椅 / 賣點:久坐不腰痛、可調腰靠")
print(out)

## 5. Business rule vs. Model rule:哪些約束不可妥協?

寫 spec 最核心的判斷,是區分兩種約束:

| 類型 | 性質 | 例子 | 下游去向 |
|------|------|------|----------|
| **Business rule(業務硬約束)** | 不可妥協、需可驗證、違反即失敗 | 「絕不虛構價格」「只能用提供的庫存資料」「不得輸出個資」 | 未來在 **M5 變 guardrail**、在 **M7 變驗證指標** |
| **Model rule(模型風格偏好)** | 可放寬、影響品質非正確性 | 「語氣親切」「標題盡量押韻」 | 留在 spec 文字即可 |

> 關鍵:**business rule 不能只靠 prompt 拜託模型遵守**——它最終要落到程式層的 guardrail / 驗證(M5、M7)。spec 是把這些硬約束**明確寫下來**的地方,讓它們後續可以被機器檢查。

In [ ]:
# 把 business rule 與 model rule 分開標註,讓下游(M5 guardrail / M7 eval)能對應
spec_v2 = build_spec(
    role="你是電商客服助理。",
    goal="回答顧客關於訂單與退換貨的問題。",
    inputs="顧客問題 + 系統提供的訂單資料(JSON)。",
    constraints=(
        "[業務硬約束] 只能根據提供的訂單資料回答,無資料時明說『查不到』,絕不虛構;不得輸出顧客個資全文。\n"
        "[模型偏好] 語氣親切、精簡(Be Concise)、用繁體中文。"
    ),
    output_format='回傳 JSON:{"answer": "...", "used_order_fields": ["..."]}',
    success_criteria="answer 不含未提供的事實;查無資料時 used_order_fields 為空且明說查不到。",
)
print(spec_v2)

## 6. 同一份 spec 跨模型一致性(Claude / Gemini)

好的 spec 應該是**模型無關**的——換 provider 仍得到結構一致的輸出。這是「收斂」的最佳證明。

In [ ]:
# Claude
try:
    import anthropic
    claude = anthropic.Anthropic()
    msg = claude.messages.create(
        model="claude-sonnet-4-6", max_tokens=512, system=spec,
        messages=[{"role": "user", "content": "產品:人體工學椅 / 賣點:久坐不腰痛、可調腰靠"}],
    )
    print("Claude:", msg.content[0].text)
except Exception as e:
    print("跳過 Claude:", type(e).__name__, e)

In [ ]:
# Gemini(用 schema 強制 JSON 結構)
try:
    from google import genai
    from google.genai import types
    gem = genai.Client()
    resp = gem.models.generate_content(
        model="gemini-2.5-flash",
        contents="產品:人體工學椅 / 賣點:久坐不腰痛、可調腰靠",
        config=types.GenerateContentConfig(
            system_instruction=spec,
            response_mime_type="application/json",
        ),
    )
    print("Gemini:", resp.text)
except Exception as e:
    print("跳過 Gemini:", type(e).__name__, e)

## 7. 從 Prompt 到 Context 到 Harness:spec 在哪一層?

提示詞的角色隨模型能力演進,經歷三個階段:

| 階段 | 年代 | 核心技能 | 你交付什麼 |
|------|------|----------|-----------|
| **Prompt Engineering** | 2022–24 | 找對「咒語」、堆 few-shot/CoT | 一段文字指令 |
| **Context Engineering** | 2025 | 設計模型周邊的**資訊環境** | system 指令 + 檢索 + 記憶 + 工具 |
| **Harness Engineering** | 2026 | 打造 agent 的**執行環境** | 規格 + 工具 + 護欄 + 驗證迴圈 |

**spec 不是被淘汰,而是升級了定位**:它從「一段 prompt」變成 context/harness 的**核心契約**。

- 往下接 M3:spec 的「輸出格式」用 structured output **強制**。
- 往下接 M5:spec 的「business rule」變成 agent 的 **guardrail**。
- 往下接 M7:spec 的「成功標準」變成**評估指標**與回饋迴圈。

> 一句話:**寫好 spec,就是替後面每一層控制機制先打好地基。**

## 8. 練習

1. 用 `build_spec()` 為「把會議記錄整理成『決議 / 待辦 / 負責人』三欄」寫一份 spec,並標出哪些是 business rule、哪些是 model rule。
2. 把同一份 spec 餵 OpenAI 與 Claude,比較輸出結構是否一致。
3. 想一想:你寫的 business rule 之中,哪些到了 M5 應該變成程式層 guardrail?哪些到 M7 該變成驗證指標?

> 觀察重點:spec 是否讓**不同次執行、不同模型**的輸出結構穩定一致?

---

## 本章小結

1. **2026 是減法**:強模型不需長 prompt;把意圖收斂成短而結構化的 **spec(規格契約)**。
2. **七欄位 / CTCO**:role / goal / inputs / constraints / output format / success criteria / examples。
3. **區分 business rule vs model rule**:硬約束需可驗證,最終落到 M5 guardrail / M7 eval,不能只靠 prompt 拜託。
4. **好 spec 模型無關**:OpenAI / Claude / Gemini 都應產生結構一致的輸出。
5. **spec 是控制權階梯的地基**:輸出格式接 M3、業務規則接 M5、成功標準接 M7;prompt→context→harness 的演進中,spec 升級為核心契約。